# Assignment 1 — The Headline Test: run and comparison
Predictions were written in `1_attempt.md` before running this notebook.

In [1]:
import pandas as pd
df = pd.read_csv('teaching_prices_clean.csv', parse_dates=['date'])
df.head()

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## Data dictionary (columns that exist)

In [2]:
df.dtypes

date          datetime64[us]
item                     str
city                     str
price                float64
unit                     str
data_label               str
dtype: object

In [3]:
actual = {'shape': df.shape, 'distinct_items': df['item'].nunique(), 'distinct_dates': df['date'].nunique()}
actual

{'shape': (96, 6), 'distinct_items': 4, 'distinct_dates': 12}

## Prediction vs actual
Copy the predictions from `1_attempt.md` below.

In [4]:
predicted = {'shape': (97, 6), 'distinct_items': 4, 'distinct_dates': 12}  # from 1_attempt.md
pd.DataFrame({'predicted': predicted, 'actual': actual})

,predicted,actual
shape,"(97, 6)","(96, 6)"
distinct_items,4,4
distinct_dates,12,12


## What else is in the data

In [5]:
print('items:', sorted(df['item'].unique()))
print('cities:', sorted(df['city'].unique()))
print('dates:', df['date'].min().date(), '->', df['date'].max().date())
print('units:', df['unit'].unique())
print('label:', df['data_label'].unique())

items: ['Oil B', 'Pulse C', 'Staple A', 'Sweetener D']
cities: ['City 1', 'City 2']
dates: 2026-01-01 -> 2026-03-19
units: <StringArray>
['PKR per kg (illustrative)']
Length: 1, dtype: str
label: <StringArray>
['ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS']
Length: 1, dtype: str


In [6]:
wide = df.groupby(['item','date'])['price'].mean().unstack('date')
change = (wide.iloc[:, -1] / wide.iloc[:, 0] - 1) * 100
change.round(1).rename('% change first->last date')

item
Oil B          15.3
Pulse C         3.7
Staple A       13.0
Sweetener D    19.4
Name: % change first->last date, dtype: float64

## Testing my two questions (Jan–Mar average, within tolerance)

In [7]:
def test(item, city, target, tol):
    sub = df[(df.item == item) & (df.city == city) & (df.date.dt.month <= 3)]
    if sub.empty:
        return f'{item} / {city}: NO ROWS - cannot test'
    avg = sub.price.mean()
    ok = abs(avg - target) <= tol * target
    return f'{item} / {city}: Jan-Mar avg = {avg:.1f} vs {target} ±{tol:.0%} -> {"YES" if ok else "NO"} ({len(sub)} rows, dates {sub.date.min().date()}..{sub.date.max().date()})'

print(test('Staple A', 'City 2', 350, 0.10))
print(test('Oil B', 'City 1', 550, 0.15))
print(test('Staple A', 'City B', 350, 0.10))  # the claim as originally heard

Staple A / City 2: Jan-Mar avg = 114.2 vs 350 ±10% -> NO (12 rows, dates 2026-01-01..2026-03-19)
Oil B / City 1: Jan-Mar avg = 321.8 vs 550 ±15% -> NO (12 rows, dates 2026-01-01..2026-03-19)
Staple A / City B: NO ROWS - cannot test


In [8]:
# My extra price-level guesses vs the whole-period average
df.groupby(['item','city']).price.mean().round(1).unstack()

city,City 1,City 2
item,,
Oil B,321.8,327.8
Pulse C,202.9,208.9
Staple A,108.2,114.2
Sweetener D,163.8,169.8
